In [22]:
import wbgapi as wb
import pandas as pd
import sys
import os

# Get the absolute path of the directory containing this nb
current_dir = os.getcwd()

project_root = os.path.abspath(os.path.join(current_dir, '../..'))

if project_root not in sys.path:
    sys.path.append(project_root)

from src.constants import countries, years


# Data sourced from the World Bank Development Indicators (WDI)

# Define countries (think we will actually just use the entire world, but let's start with a subset of major economies)
countries = countries
years = years

# Define what we are interested in
# co2 intensity, renewable %, political stability, gdp per capita, energy use per capita


indicators = {
    'EG.USE.PCAP.KG.OE': 'energy_use_per_capita',
    'NY.GDP.MKTP.KD': 'gdp_constant_usd',
    'EG.ELC.RNWX.ZS': 'renewable_electricity_pct',
    'NY.GDP.PCAP.KD': 'gdp_per_capita_constant',
    'EG.FEC.RNEW.ZS': 'renewable_final_energy_pct',
    'EG.ELC.COAL.ZS': 'coal_electricity_pct' ,
    'EN.ATM.CO2E.PP.GD': 'co2_emissions_per_gdp',

}

# Fetch data
df_wb = wb.data.DataFrame(indicators.keys(), countries, time=years, labels=False)

# Rename the columns

df_wb = df_wb.rename(columns=indicators)

# Set display format 
pd.options.display.float_format = '{:,.2f}'.format

# Now print data (head is enough to check if it looks correct)
print(df_wb.head())

                                      YR2015             YR2016  \
economy series                                                    
ARG     EG.ELC.COAL.ZS                  2.01               1.61   
        EG.ELC.RNWX.ZS                 -3.80              -2.94   
        EG.FEC.RNEW.ZS                  9.40               9.40   
        EG.USE.PCAP.KG.OE           1,919.76           1,889.87   
        NY.GDP.MKTP.KD    594,749,285,413.21 582,376,550,428.09   

                                      YR2017             YR2018  \
economy series                                                    
ARG     EG.ELC.COAL.ZS                  1.55               1.56   
        EG.ELC.RNWX.ZS                 -3.09              -2.69   
        EG.FEC.RNEW.ZS                 10.40              10.50   
        EG.USE.PCAP.KG.OE           1,874.83           1,806.78   
        NY.GDP.MKTP.KD    598,790,850,843.68 583,118,120,294.00   

                                      YR2019             YR2

In [21]:
# Reset index to turn 'economy' and 'series' back into columns
df_clean = df_wb.reset_index()

# Rename columns
df_clean['series'] = df_clean['series'].map(indicators)

# Melt into long format
df_melted = df_clean.melt(id_vars=['economy', 'series'], 
                          var_name='year', 
                          value_name='value')

# Pivot so that each indicator becomes a column
df_final = df_melted.pivot_table(index=['economy', 'year'], 
                                 columns='series', 
                                 values='value').reset_index()

# Clean up year strings
df_final['year'] = df_final['year'].str.replace('YR', '').astype(int)

# Renaming
df_final.rename(columns={'economy': 'country_code'}, inplace=True)

print(df_final.head())

# Save as csv to data processed folder
df_final.to_csv('../../data/processed/world_bank_indicators_targeted.csv', index=False)


series country_code  year  coal_electricity_pct  energy_use_per_capita  \
0               ARG  2015                  2.01               1,919.76   
1               ARG  2016                  1.61               1,889.87   
2               ARG  2017                  1.55               1,874.83   
3               ARG  2018                  1.56               1,806.78   
4               ARG  2019                  0.92               1,777.42   

series   gdp_constant_usd  gdp_per_capita_constant  renewable_electricity_pct  \
0      594,749,285,413.21                13,679.63                      -3.80   
1      582,376,550,428.09                13,265.89                      -2.94   
2      598,790,850,843.68                13,520.11                      -3.09   
3      583,118,120,294.00                13,058.33                      -2.69   
4      571,450,737,224.44                12,706.40                       1.50   

series  renewable_final_energy_pct  
0                             9